# Shreve Week 07 — Multivariable Stochastic Calculus

**Week 7** — Multivariable Stochastic Calculus

This notebook teaches **multivariable stochastic calculus** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **Multi-dimensional Brownian motion** | Ch. 4.6 |
| 2 | **Itô lemma in \(n\) dimensions** | Ch. 4.6 |
| 3 | **Correlation & covariance** | Ch. 4.6 |
| 4 | **Two-asset model** | Ch. 4.6 |
| — | **Problem set** | Ch. 4 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 4 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Multidimensional Brownian Motion

\(W_t = (W_t^{(1)}, \ldots, W_t^{(d)})\) with independent components, \(E[W_t^{(i)} W_t^{(j)}] = \min(t,s)\, \delta_{ij}\).

**Shreve Ch. 4.6:** vector Brownian motion.


In [ ]:
rng = np.random.default_rng(60)
T, n = 1.0, 1000
dt = T / n
dW = rng.normal(0, np.sqrt(dt), size=(n, 2))
W = np.cumsum(dW, axis=0)
corr = np.corrcoef(W[:, 0], W[:, 1])[0, 1]
print(f"Corr(W¹, W²) at T = {corr:.4f} (theory 0)")


---
# Part 2 — Itô's Lemma (Several Variables)

For \(f(t, X_t^{(1)}, \ldots, X_t^{(n)})\) with \(dX^{(i)} = \mu_i dt + \sum_j \sigma_{ij} dW^{(j)}\):

$$df = \frac{\partial f}{\partial t}dt + \sum_i \frac{\partial f}{\partial x_i} dX^{(i)} + \frac{1}{2}\sum_{i,j}\frac{\partial^2 f}{\partial x_i \partial x_j} d[X^{(i)}, X^{(j)}]$$

**Shreve Ch. 4.6:** multi-dimensional Itô.


In [ ]:
# f(W1, W2) = W1 * W2: df = W2 dW1 + W1 dW2 (no dt term)
rng = np.random.default_rng(61)
dt = 1/1000
dW1 = rng.normal(0, np.sqrt(dt), 1000)
dW2 = rng.normal(0, np.sqrt(dt), 1000)
W1 = np.cumsum(dW1)
W2 = np.cumsum(dW2)
f = W1 * W2
df = f[1:] - f[:-1]
df_ito = W2[:-1]*dW1 + W1[:-1]*dW2
print(f"Corr(df, Itô approx) = {np.corrcoef(df, df_ito)[0,1]:.4f}")


---
# Part 3 — Correlated Brownian Motions

Build \(B_1 = W_1\), \(B_2 = \rho W_1 + \sqrt{1-\rho^2} W_2\) from independent \(W_1, W_2\).

**Shreve Ch. 4.6:** correlation structure.


In [ ]:
rho = 0.7
rng = np.random.default_rng(62)
n = 200_000
Z1 = rng.standard_normal(n)
Z2 = rng.standard_normal(n)
B1 = Z1
B2 = rho*Z1 + np.sqrt(1-rho**2)*Z2
print(f"Corr(B1,B2) = {np.corrcoef(B1,B2)[0,1]:.4f} (ρ={rho})")


---
# Part 4 — Two Correlated Stocks

\(dS_1 = \mu_1 S_1 dt + \sigma_1 S_1 dB_1\), \(dS_2 = \mu_2 S_2 dt + \sigma_2 S_2 dB_2\) with correlated \(B_1, B_2\).

**Shreve Ch. 4.6:** multi-asset GBM (preview of Ch. 5).


In [ ]:
def simulate_two_stocks(S01, S02, mu1, mu2, sig1, sig2, rho, T, n, seed=63):
    rng = np.random.default_rng(seed)
    dt = T / n
    Z1 = rng.standard_normal(n)
    Z2 = rng.standard_normal(n)
    dB1 = np.sqrt(dt) * Z1
    dB2 = np.sqrt(dt) * (rho*Z1 + np.sqrt(1-rho**2)*Z2)
    S1 = S01 * np.exp(np.cumsum((mu1-0.5*sig1**2)*dt + sig1*dB1))
    S2 = S02 * np.exp(np.cumsum((mu2-0.5*sig2**2)*dt + sig2*dB2))
    return S1, S2

S1, S2 = simulate_two_stocks(100, 100, 0.08, 0.06, 0.2, 0.25, 0.6, 1, 252)
print(f"Return corr = {np.corrcoef(np.diff(np.log(S1)), np.diff(np.log(S2)))[0,1]:.4f}")


**Your turn:** Derive \(d(S_1 S_2)\) for correlated GBMs using multi-dimensional Itô.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Write Itô's lemma for \(f(W_t^{(1)}, W_t^{(2)}) = W^{(1)} W^{(2)}\).

2. How do you construct Brownian motions with correlation matrix \(\Sigma\)?

3. For two stocks with correlation \(\rho\), find \(d(S_1 S_2)\).

4. State the multidimensional Itô formula for \(n\) processes.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(df = W^{(2)} dW^{(1)} + W^{(1)} dW^{(2)}\) (no \(dt\) since independent).

**2.** Cholesky: \(\Sigma = L L^\top\), \(B = L Z\) for standard normal \(Z\).

**3.** Product rule + \((dS_1)(dS_2) = \rho \sigma_1 \sigma_2 S_1 S_2 dt\).

**4.** See Shreve Ch. 4.6 Eq. with \(d[X^{(i)},X^{(j)}]\) terms.

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 4 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
